## Chunk length

In [ ]:
with open("hubspot_blog_post.txt", "r",encoding="utf-8") as f:
    text = f.read()
chunks = [text[i : i + 200] for i in range(0, len(text), 200)]
for chunk in chunks:
    print("-" * 20)
    print(chunk)

## Sliding chunk

In [ ]:
def sliding_window(text, window_size, step_size):
    if window_size > len(text) or step_size < 1:
        return []
    return [text[i:i+window_size] for i
    in range(0, len(text) - window_size + 1, step_size)]
    
text = "This is an example of sliding window text chunking made by Prompting book and revised by Pedro Delgado"
window_size = 20
step_size = 5

chunks = sliding_window(text, window_size, step_size)
for idx, chunk in enumerate(chunks):
    print(f"Chunk {idx + 1}: {chunk}")

Chunk 1: This is an example o
Chunk 2: is an example of sli
Chunk 3:  example of sliding 
Chunk 4: ple of sliding windo
Chunk 5: f sliding window tex
Chunk 6: ding window text chu
Chunk 7: window text chunking
Chunk 8: w text chunking made
Chunk 9: t chunking made by P
Chunk 10: nking made by Prompt
Chunk 11:  made by Prompting b
Chunk 12:  by Prompting book a
Chunk 13: rompting book and re
Chunk 14: ing book and revised
Chunk 15: ook and revised by P
Chunk 16: nd revised by Pedro 
Chunk 17: vised by Pedro Delga


## Tiktoken

In [ ]:
# 1. Import the package:
import tiktoken
# 2. Load an encoding with tiktoken.get_encoding()
encoding = tiktoken.get_encoding("cl100k_base")

In [ ]:
# 3. Turn some text into tokens with encoding.encode()
# while turning tokens into text with encoding.decode()
print(encoding.encode("Learning how to use Tiktoken is fun!"))
print(encoding.decode([1061, 15009, 374, 264, 2294, 1648,
311, 4048, 922, 15592, 0]))
# [48567, 1268, 311, 1005, 73842, 5963, 374, 2523, 0]
# "Data engineering is a great way to learn about AI!"

In [6]:
def count_tokens(text_string: str, encoding_name: str) -> int:
    """
    Returns the number of tokens in a text string using a given encoding.
    Args:
    text: The text string to be tokenized.
    encoding_name: The name of the encoding to be used for tokenization.
    Returns:
    The number of tokens in the text string.
    Raises:
    ValueError: If the encoding name is not recognized.
    """
    encoding = tiktoken.get_encoding(encoding_name)
    num_tokens = len(encoding.encode(text_string))
    return num_tokens

In [7]:
# 4. Use the function to count the number of tokens in a text string.
text_string = "Hello world! This is a test."
print(count_tokens(text_string, "cl100k_base"))

8


## Token call ChatGPT

In [10]:
def num_tokens_from_messages(messages, model="gpt-3.5-turbo-0613"):
    """Return the number of tokens used by a list of messages."""
    try:
        encoding = tiktoken.encoding_for_model(model)
    except KeyError:
        print("Warning: model not found. Using cl100k_base encoding.")
        encoding = tiktoken.get_encoding("cl100k_base")
    if model in {
        "gpt-3.5-turbo-0613",
        "gpt-3.5-turbo-16k-0613",
        "gpt-4-0314",
        "gpt-4-32k-0314",
        "gpt-4-0613",
        "gpt-4-32k-0613",
        }:
        tokens_per_message = 3
        tokens_per_name = 1
    elif model == "gpt-3.5-turbo-0301":
        tokens_per_message = 4 # every message follows
        # <|start|>{role/name}\n{content}<|end|>\n
        tokens_per_name = -1 # if there's a name, the role is omitted
    elif "gpt-3.5-turbo" in model:
        print('''Warning: gpt-3.5-turbo may update over time. Returning
        num tokens assuming gpt-3.5-turbo-0613.''')
        return num_tokens_from_messages(messages, model="gpt-3.5-turbo-0613")
    elif "gpt-4" in model:
        print('''Warning: gpt-4 may update over time.
        Returning num tokens assuming gpt-4-0613.''')
        return num_tokens_from_messages(messages, model="gpt-4-0613")
    else:
        raise NotImplementedError(
            f"""num_tokens_from_messages() is not implemented for model
            {model}."""
        )
    num_tokens = 0
    for message in messages:
        num_tokens += tokens_per_message
        for key, value in message.items():
            num_tokens += len(encoding.encode(value))
            if key == "name":
                num_tokens += tokens_per_name
    num_tokens += 3 # every reply is primed with
    # <|start|>assistant<|message|>
    return num_tokens

In [11]:
example_messages = [
{
"role": "system",
"content": '''You are a helpful, pattern-following assistant that
translates corporate jargon into plain English.''',
},
{
"role": "system",
"name": "example_user",
"content": "New synergies will help drive top-line growth.",
},
{
"role": "system",
"name": "example_assistant",
"content": "Things working well together will increase revenue.",
},
{
"role": "system",
"name": "example_user",
"content": '''Let's circle back when we have more bandwidth to touch
base on opportunities for increased leverage.''',
},
{
"role": "system",
"name": "example_assistant",
"content": '''Let's talk later when we're less busy about how to
do better.''',
},
{
"role": "user",
"content": '''This late pivot means we don't have
time to boil the ocean for the client deliverable.''',
},
]

In [12]:
for model in ["gpt-3.5-turbo-0301", "gpt-4-0314"]:
    print(model)
    # example token count from the function defined above
    print(f'''{num_tokens_from_messages(example_messages, model)}
    prompt tokens counted by num_tokens_from_messages().''')

gpt-3.5-turbo-0301
132
    prompt tokens counted by num_tokens_from_messages().
gpt-4-0314
134
    prompt tokens counted by num_tokens_from_messages().


## Majority Vote for classification

In [3]:
from openai import OpenAI
import os


In [5]:
client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))
base_template = """
Given the statement, classify it as either "Compliment", "Complaint", or
"Neutral":
1. "The sun is shining." - Neutral
2. "Your support team is fantastic!" - Compliment
3. "I had a terrible experience with your software." - Complaint
You must follow the following principles:
- Only return the single classification word. The response should be either
"Compliment", "Complaint", or "Neutral".
- Perform the classification on the text enclosed within ''' delimiters.
'''{content}'''
Classification:
"""

In [14]:
responses = []
for i in range(0, 3):
    response = client.chat.completions.create(
        model="gpt-3.5",
        messages=[{"role": "system",
            "content": base_template.format(content='''Outside is rainy, but I am having a great day, I just don't understand how people
            live, I'm so sad!'''),}],)
    responses.append(response.choices[0].message.content.strip())

NotFoundError: Error code: 404 - {'error': {'message': 'The model `gpt-3.5` does not exist or you do not have access to it.', 'type': 'invalid_request_error', 'param': None, 'code': 'model_not_found'}}

from openai import OpenAI
import os

In [11]:
def most_frequent_classification(responses):
    # Use a dictionary to count occurrences of each classification
    count_dict = {}
    for classification in responses:
        count_dict[classification] = count_dict.get(classification, 0) + 1
        
    # Return the classification with the maximum count
    return max(count_dict, key=count_dict.get)

In [12]:
print(most_frequent_classification(responses)) # Expected Output: Neutral

ValueError: max() iterable argument is empty